### Notebook inspired by https://github.com/ndb796/PyTorch-Adversarial-Attack-Baselines-for-ImageNet-CIFAR10-MNIST


### Preliminaries

In [ ]:
!git clone https://github.com/ndb796/PyTorch-Adversarial-Attack-Baselines-for-ImageNet-CIFAR10-MNIST.git

In [ ]:
%cd PyTorch-Adversarial-Attack-Baselines-for-ImageNet-CIFAR10-MNIST
!pwd

In [ ]:
import torch
import torch.nn as nn
import torchvision.transforms as T

import PIL

from urllib.request import urlretrieve
import json

# Load pretrained model from torch hub

We will be using a standard ResNet50 pre-trained on ImageNet for our experiments.

In [ ]:
model = torch.hub.load('pytorch/vision:v0.6.0', 'resnet50', pretrained=True)
model

# Prepare image utils

- preprocessing (using standard imagenet mean and std for evaluating image with NN)
  - Center Crop the image to 224 x 224 pixels
  - ToTensor: image is normalized in the 0-1 range, channel dimensions order is changed from `h x w x c` to `c x h x w`
  - Normalize: normalize tensor so that it has mean 0 and std 1
- loading util
- transformation inversion for image plotting purposes (inverting only normalization and ToTensor, we cannot invert the cropping)

In [ ]:
img_preprocessing = T.Compose([
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
  ])

def load_img(path):
    image = PIL.Image.open(path)
    return image

def invert_transform(img_tensor):
  m = torch.Tensor([0.485, 0.456, 0.406]).unsqueeze(-1).unsqueeze(-1)
  s = torch.Tensor([0.229, 0.224, 0.225]).unsqueeze(-1).unsqueeze(-1)
  img_numpy = (s * img_tensor + m) * 255
  img_numpy = img_numpy.permute(1,2,0)
  img_numpy = img_numpy.numpy()
  return img_numpy.astype("uint8")

let's try it out on an image... notice the folder structure

`.../class_id/file.jpeg` - the class ID is in the file name

In [ ]:
img = load_img('datasets/ILSVRC2012_img_val_subset/18/ILSVRC2012_val_00000476.JPEG')
img

corresponding label is number 18 → magpie

In [ ]:
# load 1,000 labels from the ImageNet config file
with open('datasets/imagenet.json') as f:
    imagenet_labels = json.load(f)

print(imagenet_labels[18])

let's try the transformation and inversion - of course the center crop to `224 x 224` is not invertible, we can only recover the original color space

In [ ]:
img_tensor = img_preprocessing(img)
PIL.Image.fromarray(invert_transform(img_tensor))

let's see if the NN works accordingly

In [ ]:
model.eval()
out = model(img_tensor.unsqueeze(0))
out.argmax()

The label is in the folder name:

`18/ILSVRC2012_val_00000476.JPEG`

We create a label as a `long` type torch tensor. This will be useful when generating the adversarial attack

In [ ]:
label = torch.Tensor([18]).long()
label

# Generate adversarial attack using FGSM

Generate an adversarial attack using FGSM with a radius $\delta$ of 0.03

$$
\text{FGSM}(x, y, \delta) = x + \delta \cdot \text{sign}(\nabla CE_x(f(x), y))
$$

1. Create a standard PyTorch training loop with only one iteration and one image, but no optimizer. It should be a **training** loop, since we need the gradients, but the model should be in `.eval()` mode since we are not training its parameters
2. Create a copy of the image before starting the training loop
3. Add `.requires_grad=True` to the image before the training loop
4. Use the information provided by the gradient to compute the FGSM attack

In [ ]:
model.eval()

delta = .03

# your code here

img_tensor_copy = img_tensor.clone()

# ...

adversarial_example = # ...

Let's see if the adversarial attack is perceptually similar to the initial image...

In [ ]:
PIL.Image.fromarray(invert_transform(adversarial_example.detach()))

...and finally, as a check, let's see if the prediction of the model on the adversarial example has changed...

In [ ]:
new_prediction = model(adversarial_example.unsqueeze(0)).argmax().item()
print(new_prediction)

print(imagenet_labels[new_prediction])

# For the bravest: implement PGD

PGD is FGSM applied for $k$ steps, with a small radius $\alpha$.
After each application of FGSM, the value of the adversarial example is _clamped_ in the interval $[x-\delta, x+\delta]$, where $x$ is the original image.

Implement PGD with $k=10$, $\delta=0.03$, and $\alpha=0.005$.

In [ ]:
# your code here